# Unidad 4 · Colab 1 de 3
## Protocolo HTTP y consumo de APIs con requests

**Objetivos de este notebook**

- Entender el protocolo HTTP: métodos, cabeceras, parámetros y códigos de respuesta.
- Consumir APIs públicas y privadas con la librería `requests`.
- Autenticarte con API Keys, tokens Bearer y entender el flujo de OAuth2.
- Manejar errores, timeouts y sesiones de forma robusta.

> **Nivel:** intermedio. Se asume Python básico y algo de experiencia previa usando `requests` o similar.

**Herramientas de práctica:** [httpbin.org](https://httpbin.org) (echo server ideal para ver exactamente qué se envía y recibe), la [API REST de GitHub](https://docs.github.com/es/rest) y [PokeAPI](https://pokeapi.co/docs/v2) (sin autenticación).

---

## 1. El protocolo HTTP

HTTP es un protocolo de request/response: el cliente (tu script, el navegador) envía una solicitud a una URL, y el servidor devuelve una respuesta con un código de estado, cabeceras y (generalmente) un cuerpo.

Documentación oficial: [Overview de HTTP (MDN)](https://developer.mozilla.org/es/docs/Web/HTTP/Overview)

### Métodos HTTP

| Método | Uso típico | ¿Body? | ¿Idempotente? |
|---|---|---|---|
| `GET` | Leer un recurso | No | Sí |
| `POST` | Crear un recurso | Sí | No |
| `PUT` | Reemplazar un recurso completo | Sí | Sí |
| `PATCH` | Modificar parcialmente un recurso | Sí | No |
| `DELETE` | Eliminar un recurso | Opcional | Sí |

*Idempotente* significa que ejecutar la misma operación varias veces produce el mismo resultado que ejecutarla una sola vez (ej. `DELETE` sobre algo que ya no existe no cambia nada más).

Documentación oficial: [Métodos HTTP (MDN)](https://developer.mozilla.org/es/docs/Web/HTTP/Methods)

### Cabeceras y parámetros

- **Cabeceras (headers):** metadatos de la request/response. Ejemplos: `Content-Type` (formato del body), `Authorization` (credenciales), `Accept` (formato que el cliente espera recibir).
- **Query params:** van en la URL después de `?`, para filtrar/paginar (`?page=2&limit=10`).
- **Path params:** forman parte de la ruta (`/usuarios/42`).
- **Body:** el contenido de la solicitud (típicamente JSON) en `POST`/`PUT`/`PATCH`.

Documentación oficial: [Cabeceras HTTP (MDN)](https://developer.mozilla.org/es/docs/Web/HTTP/Headers)

### Códigos de respuesta

| Rango | Significado | Ejemplos |
|---|---|---|
| `2xx` | Éxito | `200 OK`, `201 Created`, `204 No Content` |
| `3xx` | Redirección | `301 Moved Permanently`, `304 Not Modified` |
| `4xx` | Error del cliente | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `422 Unprocessable Entity`, `429 Too Many Requests` |
| `5xx` | Error del servidor | `500 Internal Server Error`, `503 Service Unavailable` |

Documentación oficial: [Códigos de estado HTTP (MDN)](https://developer.mozilla.org/es/docs/Web/HTTP/Status)

### Ejercicio 1 — Elegir método y código de respuesta

Para cada escenario, indicá qué método HTTP usarías y qué código de respuesta esperarías si todo sale bien:

1. Registrar un nuevo usuario.
2. Obtener el perfil de un usuario por su id.
3. Actualizar solo el email de un usuario.
4. Borrar una publicación.
5. Un usuario intenta acceder a un recurso sin estar autenticado.

<details>
<summary>💡 Ver solución</summary>

1. `POST /usuarios` → `201 Created`
2. `GET /usuarios/{id}` → `200 OK`
3. `PATCH /usuarios/{id}` → `200 OK` (o `204 No Content` si no devuelve body)
4. `DELETE /publicaciones/{id}` → `204 No Content`
5. Cualquier método → `401 Unauthorized`

</details>

## 2. Consumir APIs con `requests`

```python
import requests

resp = requests.get('https://httpbin.org/get', params={'ciudad': 'Rosario'})
print(resp.status_code)
print(resp.json())
```

Documentación oficial: [requests](https://requests.readthedocs.io/en/latest/)

In [1]:
import requests

resp = requests.get('https://httpbin.org/get', params={'ciudad': 'Rosario'})
print(resp.status_code)
print(resp.json())

200
{'args': {'ciudad': 'Rosario'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, br, zstd', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.32.4', 'X-Amzn-Trace-Id': 'Root=1-6aa01ebf-0f85e2e507479dd223206c0b'}, 'origin': '34.34.31.73', 'url': 'https://httpbin.org/get?ciudad=Rosario'}


In [2]:
import requests

# GET con query params
resp = requests.get('https://httpbin.org/get', params={'ciudad': 'Rosario', 'pais': 'AR'})
print('GET status:', resp.status_code)
print(resp.json()['args'])

# POST con body JSON
resp = requests.post('https://httpbin.org/post', json={'nombre': 'Notebook Lenovo', 'precio': 599})
print('POST status:', resp.status_code)
print(resp.json()['json'])

GET status: 200
{'ciudad': 'Rosario', 'pais': 'AR'}
POST status: 200
{'nombre': 'Notebook Lenovo', 'precio': 599}


### Ejercicio 2 — PUT y DELETE con httpbin

httpbin.org tiene un endpoint por cada método (`/put`, `/delete`, etc.) que simplemente devuelve (hace *echo*) lo que le mandaste. Completá el código para: (1) hacer un `PUT` a `https://httpbin.org/put` con un body JSON `{precio: 549}`, y (2) hacer un `DELETE` a `https://httpbin.org/delete` con un header personalizado `X-Motivo: descontinuado`.

In [ ]:
# TODO: PUT a https://httpbin.org/put con json={'precio': 549}

# TODO: DELETE a https://httpbin.org/delete con headers={'X-Motivo': 'descontinuado'}

<details>
<summary>💡 Ver solución</summary>

```python
resp = requests.put('https://httpbin.org/put', json={'precio': 549})
print(resp.status_code, resp.json()['json'])

resp = requests.delete('https://httpbin.org/delete', headers={'X-Motivo': 'descontinuado'})
print(resp.status_code, resp.json()['headers'].get('X-Motivo'))
```

</details>

In [3]:
resp = requests.put('https://httpbin.org/put', json={'precio': 549})
print(resp.status_code, resp.json()['json'])

resp = requests.delete('https://httpbin.org/delete', headers={'X-Motivo': 'descontinuado'})
print(resp.status_code, resp.json()['headers'].get('X-Motivo'))

200 {'precio': 549}
200 descontinuado


## 3. Autenticación: API Keys, Bearer tokens y OAuth2

- **API Key:** una cadena fija que identifica tu aplicación, enviada como query param o header (`X-API-Key`).
- **Bearer token:** una credencial (a veces temporal) enviada en el header `Authorization: Bearer <token>`. Es el mecanismo típico para tokens de acceso de OAuth2 o Personal Access Tokens.
- **OAuth2:** un protocolo de autorización delegada (por ejemplo, iniciar sesión con Google). El flujo más común para apps de servidor es *Authorization Code*: el usuario autoriza en el proveedor, la app recibe un `code`, lo cambia por un `access_token`, y ese token se usa en el header `Authorization`.

Documentación oficial: [oauth.net](https://oauth.net/2/) · [Autenticación en la API de GitHub](https://docs.github.com/es/rest/authentication/authenticating-to-the-rest-api)

In [ ]:
import requests

# Sin autenticacion (limite bajo de requests por hora)
resp = requests.get('https://api.github.com/rate_limit')
print('Sin token, limite por hora:', resp.json()['rate']['limit'])

# TODO: si tenes un Personal Access Token de GitHub, probalo aca
# headers = {'Authorization': 'Bearer TU_TOKEN'}
# resp = requests.get('https://api.github.com/rate_limit', headers=headers)
# print('Con token, limite por hora:', resp.json()['rate']['limit'])

In [4]:
import requests

# Sin autenticacion (limite bajo de requests por hora)
resp = requests.get('https://api.github.com/rate_limit')
print('Sin token, limite por hora:', resp.json()['rate']['limit'])

# TODO: si tenes un Personal Access Token de GitHub, probalo aca
# headers = {'Authorization': 'Bearer TU_TOKEN'}
# resp = requests.get('https://api.github.com/rate_limit', headers=headers)
# print('Con token, limite por hora:', resp.json()['rate']['limit'])

Sin token, limite por hora: 60


### Ejercicio 3 — Consumir un endpoint autenticado

Usando la API de GitHub, escribí una función `mis_repos(usuario, token=None)` que devuelva la lista de nombres de repositorios públicos de un usuario (`GET /users/{usuario}/repos`), enviando el `token` en el header `Authorization` solo si se pasó uno.

<details>
<summary>💡 Ver solución</summary>

```python
def mis_repos(usuario, token=None):
    headers = {}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    resp = requests.get(f'https://api.github.com/users/{usuario}/repos', headers=headers)
    resp.raise_for_status()
    return [repo['name'] for repo in resp.json()]

mis_repos('octocat')
```

</details>

In [5]:
def mis_repos(usuario, token=None):
    headers = {}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    resp = requests.get(f'https://api.github.com/users/{usuario}/repos', headers=headers)
    resp.raise_for_status()
    return [repo['name'] for repo in resp.json()]

mis_repos('octocat')

['boysenberry-repo-1',
 'git-consortium',
 'hello-worId',
 'Hello-World',
 'linguist',
 'octocat.github.io',
 'Spoon-Knife',
 'test-repo1']

## 4. Manejo de errores, timeouts y `Session`

```python
import requests

try:
    resp = requests.get('https://httpbin.org/delay/3', timeout=2)
except requests.exceptions.Timeout:
    print('Timeout: el servidor tardo demasiado')

# Session reutiliza la conexion TCP y headers comunes entre requests
sesion = requests.Session()
sesion.headers.update({'Authorization': 'Bearer TU_TOKEN'})
resp = sesion.get('https://httpbin.org/get')
```

Usar `Session` es más eficiente cuando hacés varios requests seguidos al mismo servicio: reutiliza la conexión y evita repetir headers en cada llamada.

In [6]:
import requests

try:
    resp = requests.get('https://httpbin.org/delay/3', timeout=2)
except requests.exceptions.Timeout:
    print('Timeout: el servidor tardo demasiado')

# Session reutiliza la conexion TCP y headers comunes entre requests
sesion = requests.Session()
sesion.headers.update({'Authorization': 'Bearer TU_TOKEN'})
resp = sesion.get('https://httpbin.org/get')

Timeout: el servidor tardo demasiado


## Mini-proyecto: cliente de una API pública

Usando [PokeAPI](https://pokeapi.co/docs/v2) (no requiere autenticación):

1. Escribí una función `obtener_pokemon(nombre)` que haga `GET https://pokeapi.co/api/v2/pokemon/{nombre}` y devuelva un diccionario con `nombre`, `peso`, `altura` y la lista de sus `tipos`.
2. Manejá el caso en que el pokemon no exista (`404`) devolviendo `None` en vez de lanzar una excepción.
3. Armá una lista con los datos de 10 pokemones distintos, reutilizando una `Session`.

**Entregable:** función `obtener_pokemon` + lista de 10 resultados, con manejo de error para nombres inválidos.

---

**Seguís en:** *Colab 2 — Desarrollo de APIs RESTful con FastAPI/Flask y Pydantic*

In [7]:
from typing import Any, Dict, List, Optional
import requests


# =====================================================================
# 1. Función para consultar PokeAPI reutilizando una sesión
# =====================================================================
def obtener_pokemon(
    nombre: str, sesion: Optional[requests.Session] = None
) -> Optional[Dict[str, Any]]:
    """Consulta la PokeAPI y devuelve los datos filtrados del pokémon.

    Retorna un diccionario con nombre, peso, altura y tipos, o None si no existe
    (404).
    """
    cliente = sesion if sesion is not None else requests
    url = f"https://pokeapi.co/api/v2/pokemon/{nombre.strip().lower()}"

    try:
        respuesta = cliente.get(url, timeout=10)

        # Si el pokemon no existe, la API retorna 404
        if respuesta.status_code == 404:
            return None

        # Lanza excepción para otros códigos de error (500, etc.)
        respuesta.raise_for_status()

        datos = respuesta.json()

        # Extracción y formateo de tipos
        tipos = [item["type"]["name"] for item in datos.get("types", [])]

        return {
            "nombre": datos.get("name"),
            "peso": datos.get("weight"),
            "altura": datos.get("height"),
            "tipos": tipos,
        }

    except requests.exceptions.RequestException as err:
        print(f"Error de red o conexión al consultar '{nombre}': {err}")
        return None


# =====================================================================
# 2. Ejecución con requests.Session() para 10 pokémones (incluyendo inválidos)
# =====================================================================
if __name__ == "__main__":
    # Lista de prueba con 10 nombres (incluyendo nombres inexistentes para testear el 404)
    candidatos = [
        "pikachu",
        "charizard",
        "bulbasaur",
        "squirtle",
        "gengar",
        "eevee",
        "snorlax",
        "lucario",
        "agumon",  # No existe en PokeAPI (404)
        "mewtwo",
    ]

    resultados: List[Optional[Dict[str, Any]]] = []

    print("=" * 65)
    print("CONSULTANDO POKEAPI REUTILIZANDO SESSION HTTP")
    print("=" * 65)

    with requests.Session() as sesion_http:
        for nombre in candidatos:
            info = obtener_pokemon(nombre, sesion=sesion_http)
            resultados.append(info)

            if info is not None:
                print(
                    f"✔ {info['nombre'].capitalize():<12} | "
                    f"Altura: {info['altura']:<3} | "
                    f"Peso: {info['peso']:<4} | "
                    f"Tipos: {', '.join(info['tipos'])}"
                )
            else:
                print(
                    f"✖ '{nombre}': No encontrado (HTTP 404) -> Retornado: None"
                )

    print("\n" + "=" * 65)
    print(f"LISTA FINAL CON LOS {len(resultados)} RESULTADOS OBTENIDOS:")
    print("=" * 65)
    import pprint

    pprint.pprint(resultados, width=80, compact=False)

CONSULTANDO POKEAPI REUTILIZANDO SESSION HTTP
✔ Pikachu      | Altura: 4   | Peso: 60   | Tipos: electric
✔ Charizard    | Altura: 17  | Peso: 905  | Tipos: fire, flying
✔ Bulbasaur    | Altura: 7   | Peso: 69   | Tipos: grass, poison
✔ Squirtle     | Altura: 5   | Peso: 90   | Tipos: water
✔ Gengar       | Altura: 15  | Peso: 405  | Tipos: ghost, poison
✔ Eevee        | Altura: 3   | Peso: 65   | Tipos: normal
✔ Snorlax      | Altura: 21  | Peso: 4600 | Tipos: normal
✔ Lucario      | Altura: 12  | Peso: 540  | Tipos: fighting, steel
✖ 'agumon': No encontrado (HTTP 404) -> Retornado: None
✔ Mewtwo       | Altura: 20  | Peso: 1220 | Tipos: psychic

LISTA FINAL CON LOS 10 RESULTADOS OBTENIDOS:
[{'altura': 4, 'nombre': 'pikachu', 'peso': 60, 'tipos': ['electric']},
 {'altura': 17,
  'nombre': 'charizard',
  'peso': 905,
  'tipos': ['fire', 'flying']},
 {'altura': 7, 'nombre': 'bulbasaur', 'peso': 69, 'tipos': ['grass', 'poison']},
 {'altura': 5, 'nombre': 'squirtle', 'peso': 90, 'tipos': 